# 📚 Exercícios — Raciocínio Baseado em Casos (RBC / CBR)

**Disciplina:** Inteligência Artificial | **Nível:** Intermediário

> Implemente o ciclo CBR (Retrieve, Reuse, Revise, Retain) com exemplos práticos.


## 1. Base de Casos — Diagnóstico de TI

In [ ]:
import math

# Base de casos: problemas de TI e suas soluções
base_casos = [
    {"id":1, "cpu_alta":1,"memoria_alta":1,"disco_cheio":0,"rede_lenta":0,"solucao":"Matar processos pesados e reiniciar serviços"},
    {"id":2, "cpu_alta":0,"memoria_alta":1,"disco_cheio":1,"rede_lenta":0,"solucao":"Liberar disco e aumentar swap"},
    {"id":3, "cpu_alta":0,"memoria_alta":0,"disco_cheio":0,"rede_lenta":1,"solucao":"Verificar driver de rede e cabo"},
    {"id":4, "cpu_alta":1,"memoria_alta":0,"disco_cheio":0,"rede_lenta":1,"solucao":"Verificar vírus e otimizar rede"},
    {"id":5, "cpu_alta":1,"memoria_alta":1,"disco_cheio":1,"rede_lenta":0,"solucao":"Reinstalar sistema operacional"},
    {"id":6, "cpu_alta":0,"memoria_alta":0,"disco_cheio":1,"rede_lenta":0,"solucao":"Excluir arquivos temporários"},
    {"id":7, "cpu_alta":0,"memoria_alta":1,"disco_cheio":0,"rede_lenta":1,"solucao":"Reiniciar servidor e verificar DNS"},
]

FEATURES = ["cpu_alta", "memoria_alta", "disco_cheio", "rede_lenta"]

def similaridade_cosseno_casos(caso1, caso2):
    v1 = [caso1[f] for f in FEATURES]
    v2 = [caso2[f] for f in FEATURES]
    num = sum(a*b for a,b in zip(v1,v2))
    den = math.sqrt(sum(a**2 for a in v1)) * math.sqrt(sum(b**2 for b in v2))
    return num/den if den > 0 else 0

def recuperar(novo_caso, base, top_k=3):
    """Retrieve: encontra os k casos mais similares."""
    sims = []
    for caso in base:
        sim = similaridade_cosseno_casos(novo_caso, caso)
        sims.append((sim, caso))
    sims.sort(key=lambda x: -x[0])
    return sims[:top_k]

# Novo problema
novo_problema = {"cpu_alta":1,"memoria_alta":0,"disco_cheio":0,"rede_lenta":1}

print("=== RBC — Diagnóstico de TI ===")
print(f"Novo problema: {novo_problema}\n")
print("Casos mais similares:")
similares = recuperar(novo_problema, base_casos, top_k=3)
for rank,(sim,caso) in enumerate(similares):
    print(f"  #{rank+1} (sim={sim:.3f}): {caso}")
    print(f"         → Solução sugerida: {caso['solucao']}\n")

# Reutilizar: adotar a solução do caso mais similar
print(f"SOLUÇÃO PROPOSTA: {similares[0][1]['solucao']}")


### 📝 Exercício 1

Adicione **3 novos casos** à base e crie um novo problema de teste. Compare as soluções sugeridas antes e depois da expansão da base.

In [ ]:
base_expandida = list(base_casos) + [
    # ✏️ Adicione 3 novos casos
    {"id":8, "cpu_alta":0,"memoria_alta":0,"disco_cheio":0,"rede_lenta":0,"solucao":"Sistema normal — nenhuma ação necessária"},
    # ...
]

meu_problema = {"cpu_alta":0, "memoria_alta":1, "disco_cheio":0, "rede_lenta":0}
similares_ex = recuperar(meu_problema, base_expandida, top_k=3)
for sim, caso in similares_ex:
    print(f"  sim={sim:.3f}: {caso['solucao']}")


## 2. RBC com Pesos nas Features

In [ ]:
def similaridade_ponderada(caso1, caso2, pesos):
    """Similaridade com pesos diferentes para cada feature."""
    num = sum(pesos[f]*caso1[f]*caso2[f] for f in FEATURES)
    den = math.sqrt(sum(pesos[f]*caso1[f]**2 for f in FEATURES)) *           math.sqrt(sum(pesos[f]*caso2[f]**2 for f in FEATURES))
    return num/den if den > 0 else 0

def recuperar_ponderado(novo_caso, base, pesos, top_k=3):
    sims = [(similaridade_ponderada(novo_caso,c,pesos),c) for c in base]
    sims.sort(key=lambda x:-x[0])
    return sims[:top_k]

# Pesos: cpu e memória são mais críticos
pesos_default  = {f:1.0 for f in FEATURES}
pesos_critico  = {"cpu_alta":3.0,"memoria_alta":2.0,"disco_cheio":1.0,"rede_lenta":1.0}

print("=== Comparação: pesos iguais vs pesos críticos ===")
print(f"Problema: {novo_problema}\n")
for nome, pesos in [("Igual", pesos_default), ("Crítico", pesos_critico)]:
    sim1, caso1 = recuperar_ponderado(novo_problema, base_casos, pesos)[0]
    print(f"[{nome}] Mais similar (sim={sim1:.3f}): {caso1['solucao']}")


## 3. Ciclo CBR Completo

Implementando Retrieve → Reuse → Revise → Retain

In [ ]:
class SistemaCBR:
    def __init__(self, base_inicial):
        self.base = list(base_inicial)
        self.log = []
    
    def retrieve(self, problema, top_k=1):
        """Fase 1: Recuperar casos similares."""
        pesos = {f:1.0 for f in FEATURES}
        sims = sorted([(similaridade_cosseno_casos(problema,c),c) for c in self.base],
                      key=lambda x:-x[0])
        return sims[:top_k]
    
    def reuse(self, casos_similares):
        """Fase 2: Reutilizar a solução do caso mais similar."""
        return casos_similares[0][1]['solucao']
    
    def revise(self, solucao_proposta, confirmado=True):
        """Fase 3: Revisar (confirmar ou corrigir a solução)."""
        if confirmado:
            return solucao_proposta
        else:
            return input("Informe a solução correta: ") if input else solucao_proposta
    
    def retain(self, problema, solucao, confirmado=True):
        """Fase 4: Reter o novo caso na base."""
        if confirmado:
            novo_caso = dict(problema)
            novo_caso['id'] = len(self.base)+1
            novo_caso['solucao'] = solucao
            self.base.append(novo_caso)
            self.log.append(f"Caso {novo_caso['id']} adicionado: {solucao}")
            return novo_caso
    
    def resolver(self, problema, solucao_real=None):
        print(f"\n--- Resolvendo: {problema} ---")
        similares = self.retrieve(problema)
        solucao = self.reuse(similares)
        print(f"Solução proposta: {solucao}")
        if solucao_real:
            confirmado = (solucao == solucao_real)
            solucao_final = solucao_real
        else:
            confirmado = True
            solucao_final = solucao
        self.retain(problema, solucao_final, confirmado)
        print(f"Solução final: {solucao_final}  ({'✅ OK' if confirmado else '🔄 Revisada'})")
        return solucao_final

cbr = SistemaCBR(base_casos)
print(f"Base inicial: {len(cbr.base)} casos")
cbr.resolver({"cpu_alta":1,"memoria_alta":0,"disco_cheio":0,"rede_lenta":1})
cbr.resolver({"cpu_alta":0,"memoria_alta":0,"disco_cheio":1,"rede_lenta":1},
             solucao_real="Verificar DNS e liberar espaço em disco")
print(f"\nBase após aprendizado: {len(cbr.base)} casos")
print("Log:", cbr.log)


### 📝 Exercício Final

Adapte o sistema CBR para um domínio de **recomendação de filmes**:
- Features: `acao`, `romance`, `comedia`, `terror`, `ficcao`
- Adicione 6 casos com filmes e seus perfis
- Resolva para um novo usuário

Qual filme seria recomendado para alguém que gosta muito de ação e ficção científica?

In [ ]:
# ✏️ Base de casos de filmes
FEATURES_FILME = ['acao','romance','comedia','terror','ficcao']

base_filmes = [
    {"id":1,"acao":1,"romance":0,"comedia":0,"terror":0,"ficcao":1,"solucao":"Recomendo: Matrix"},
    # TODO: adicione mais casos
]

def sim_filme(c1, c2):
    return similaridade_cosseno_casos(c1, c2)  # reuse sem pesos

usuario = {"acao":1,"romance":0,"comedia":0,"terror":0,"ficcao":1}
# TODO: use recuperar() com FEATURES_FILME
